# 🐍 30 Days of Python – Day 29: Building an API

## Building a Full REST API with Flask

In [ ]:
full_api_code = '''
from flask import Flask, jsonify, request
from datetime import datetime
import uuid

app = Flask(__name__)

# In-memory database
db = {
    "students": {
        "1": {
            "id": "1",
            "name": "Alice",
            "age": 22,
            "email": "alice@example.com",
            "skills": ["Python", "JavaScript"],
            "score": 88,
            "created_at": "2024-01-01"
        }
    }
}

# ─── Students CRUD ───────────────────────

@app.route("/api/v1/students", methods=["GET"])
def get_students():
    students = list(db["students"].values())
    # Query params
    city = request.args.get("city")
    min_score = request.args.get("min_score", type=int)
    if city:
        students = [s for s in students if s.get("city") == city]
    if min_score:
        students = [s for s in students if s.get("score", 0) >= min_score]
    return jsonify({"count": len(students), "students": students})

@app.route("/api/v1/students/<student_id>", methods=["GET"])
def get_student(student_id):
    student = db["students"].get(student_id)
    if not student:
        return jsonify({"error": "Student not found"}), 404
    return jsonify(student)

@app.route("/api/v1/students", methods=["POST"])
def create_student():
    data = request.get_json()
    required = ["name", "age", "email"]
    missing = [f for f in required if f not in data]
    if missing:
        return jsonify({"error": f"Missing fields: {missing}"}), 400
    student_id = str(uuid.uuid4())[:8]
    student = {
        "id": student_id,
        "created_at": datetime.now().isoformat(),
        **data
    }
    db["students"][student_id] = student
    return jsonify(student), 201

@app.route("/api/v1/students/<student_id>", methods=["PUT"])
def update_student(student_id):
    if student_id not in db["students"]:
        return jsonify({"error": "Not found"}), 404
    db["students"][student_id].update(request.get_json())
    return jsonify(db["students"][student_id])

@app.route("/api/v1/students/<student_id>", methods=["DELETE"])
def delete_student(student_id):
    if student_id not in db["students"]:
        return jsonify({"error": "Not found"}), 404
    del db["students"][student_id]
    return jsonify({"message": "Deleted successfully"})

# ─── Statistics endpoint ─────────────────
@app.route("/api/v1/stats")
def stats():
    students = list(db["students"].values())
    scores = [s.get("score", 0) for s in students]
    return jsonify({
        "total": len(students),
        "avg_score": sum(scores)/len(scores) if scores else 0,
        "max_score": max(scores) if scores else 0,
        "min_score": min(scores) if scores else 0
    })

@app.errorhandler(404)
def not_found(e):
    return jsonify({"error": "Endpoint not found"}), 404

@app.errorhandler(500)
def server_error(e):
    return jsonify({"error": "Internal server error"}), 500

if __name__ == "__main__":
    app.run(debug=True, port=5000)
'''
print(full_api_code)

## API Documentation (Swagger/OpenAPI)

In [ ]:
openapi_doc = '''
openapi: 3.0.0
info:
  title: Students API
  version: 1.0.0
  description: 30 Days of Python - Student Management API
paths:
  /api/v1/students:
    get:
      summary: Get all students
      parameters:
        - name: min_score
          in: query
          schema:
            type: integer
      responses:
        200:
          description: List of students
    post:
      summary: Create a student
      requestBody:
        required: true
        content:
          application/json:
            schema:
              type: object
              required: [name, age, email]
              properties:
                name:
                  type: string
                age:
                  type: integer
                email:
                  type: string
      responses:
        201:
          description: Student created
        400:
          description: Missing required fields
  /api/v1/students/{id}:
    get:
      summary: Get a student by ID
      parameters:
        - name: id
          in: path
          required: true
          schema:
            type: string
      responses:
        200:
          description: Student found
        404:
          description: Not found
'''
print(openapi_doc)

## Testing with pytest

In [ ]:
test_code = '''
import pytest
import json
from app import app   # import your Flask app

@pytest.fixture
def client():
    app.config['TESTING'] = True
    with app.test_client() as client:
        yield client

def test_get_students(client):
    r = client.get('/api/v1/students')
    assert r.status_code == 200
    data = json.loads(r.data)
    assert 'students' in data
    assert 'count' in data

def test_create_student(client):
    new_student = {'name': 'Test User', 'age': 25, 'email': 'test@test.com'}
    r = client.post('/api/v1/students',
                    data=json.dumps(new_student),
                    content_type='application/json')
    assert r.status_code == 201
    data = json.loads(r.data)
    assert data['name'] == 'Test User'

def test_create_student_missing_field(client):
    r = client.post('/api/v1/students',
                    data=json.dumps({'name': 'Incomplete'}),
                    content_type='application/json')
    assert r.status_code == 400
'''
print(test_code)
print('Run tests with: pytest test_app.py -v')